# Triangle Counting

## Learning Objectives

1. **Define** the global clustering coefficient in terms of triangles
2. **Implement** exact triangle counting with heavy-light decomposition
3. **Implement** approximate triangle counting via edge sampling
4. **Analyse** the complexity trade-off between exact and approximate methods

## Why Count Triangles?

A **triangle** is a set of three mutually connected nodes $\{u,v,w\}$.

**Global clustering coefficient:**
$$C = \frac{3 \times \text{number of triangles}}{\text{number of connected triples}} = \frac{3T}{\text{wedges}}$$

**Applications:**
- Spam detection: spam accounts rarely form triangles (fake connections don't know each other)
- Community quality: dense communities have many internal triangles
- Link prediction: nodes sharing many common neighbors (triangles) are likely to connect
- Network analysis: comparing real vs random graph structure

In [ ]:
from collections import defaultdict

def count_triangles_naive(adj):
    """Naive O(n * d^2) triangle counting. Only feasible for small graphs."""
    triangles = 0
    for u in adj:
        for v in adj[u]:
            if v > u:
                triangles += len(adj[u] & adj[v])
    return triangles // 2   # each triangle counted twice (once per edge)

# Small test graph
adj = defaultdict(set)
edges = [(0,1),(1,2),(2,0),(0,3),(3,1),(3,4)]
for u,v in edges:
    adj[u].add(v); adj[v].add(u)

t = count_triangles_naive(adj)
print(f"Triangles in test graph: {t}")
# Should find triangles: (0,1,2) and (0,1,3) = 2

## Heavy-Light Decomposition

**Key observation:** the naive algorithm spends most time on high-degree ("heavy") nodes.

**Heavy-light split:** threshold $\theta = \sqrt{m}$ (where $m$ = number of edges)
- **Heavy nodes:** degree > $\theta$ — at most $m/\theta = \sqrt{m}$ such nodes
- **Light nodes:** degree ≤ $\theta$

**Triangle types:**
- **Light-Light-Light:** all three vertices light → enumerate wedges from light nodes only
- **Heavy-* :** at least one heavy vertex → enumerate pairs of heavy vertices and check

**Complexity:** $O(m\sqrt{m})$ instead of $O(m \cdot d_{\max})$

In [ ]:
import math
from collections import defaultdict

def count_triangles_heavy_light(adj, edges_list):
    m = len(edges_list)
    theta = max(1, int(math.sqrt(m)))
    degree = {u: len(adj[u]) for u in adj}
    heavy = {u for u in adj if degree[u] > theta}

    # Give each node an ordering: heavy nodes first, then by degree
    order = {u: (0, degree[u], u) if u in heavy else (1, degree[u], u) for u in adj}

    triangles = 0

    # For each node u, for each pair of neighbors (v, w) where order(v) > order(u) and order(w) > order(u):
    # check if (v,w) is an edge. This avoids double-counting.
    for u in adj:
        higher_neighbors = [v for v in adj[u] if order[v] > order[u]]
        # Build set for fast lookup
        hn_set = set(higher_neighbors)
        for v in higher_neighbors:
            for w in adj[v]:
                if order[w] > order[u] and w in hn_set:
                    triangles += 1

    return triangles

import random
rng = random.Random(42)
n = 100
edge_set = set()
for i in range(n):
    for j in range(i+1,n):
        if rng.random() < 0.08:
            edge_set.add((i,j))
edges = list(edge_set)
adj2 = defaultdict(set)
for u,v in edges:
    adj2[u].add(v); adj2[v].add(u)

t_naive = count_triangles_naive(adj2)
t_hl    = count_triangles_heavy_light(adj2, edges)
print(f"Naive:       {t_naive} triangles")
print(f"Heavy-light: {t_hl} triangles")
print(f"Match: {t_naive == t_hl}")

## Approximate Triangle Counting via Sampling

**Algorithm:**
1. Sample each edge $(u,v)$ independently with probability $p$
2. For each sampled edge $(u,v)$, count common neighbors in the full graph
3. Each triangle has 3 edges; probability all 3 are sampled = $p^3$
4. Estimate: $\hat{T} = \frac{\text{observed triangles}}{p^3}$

**Error:** $\hat{T}$ is unbiased; standard deviation $\propto 1/\sqrt{\text{sample size}}$.

This works in MapReduce: sample edges in Map phase, count in Reduce.

In [ ]:
import random, math

def approx_triangles_sampling(adj, edges, p=0.3, seed=42):
    """Sample edges with prob p, count triangles, rescale."""
    rng = random.Random(seed)
    sampled = [(u,v) for u,v in edges if rng.random() < p]
    sampled_set = set()
    for u,v in sampled:
        sampled_set.add((min(u,v),max(u,v)))

    # Count triangles among sampled edges only
    sampled_adj = defaultdict(set)
    for u,v in sampled:
        sampled_adj[u].add(v); sampled_adj[v].add(u)

    t_sampled = count_triangles_naive(sampled_adj)
    return t_sampled / (p**3)

print(f"True triangles: {t_naive}")
print(f"{'p':>5}  {'estimate':>10}  {'error%':>8}")
for p in [0.5, 0.4, 0.3, 0.2]:
    estimates = [approx_triangles_sampling(adj2, edges, p=p, seed=s) for s in range(20)]
    avg = sum(estimates)/len(estimates)
    err = abs(avg - t_naive)/max(t_naive,1)*100
    print(f"{p:>5.1f}  {avg:>10.1f}  {err:>8.1f}%")

## Summary

| Method | Time | Space | Notes |
|--------|------|-------|-------|
| Naive | O(n·d²) | O(m) | Only for small graphs |
| Heavy-light | O(m√m) | O(m) | Best exact algorithm for MapReduce |
| Edge sampling | O(p·m + p³·T) | O(p·m) | Unbiased; controllable error |

For web-scale graphs (billions of edges), the sampling approach is most practical.
Heavy-light decomposition is optimal for exact counting in distributed settings.